[![Open In Colab](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module4/03-data-cleaning.ipynb)](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module4/03-data-cleaning.ipynb)

# Data Cleaning: From Messy to Analysis-Ready

**Module 4 — Data Science & Visualization** | Estimated time: 35 minutes

## Learning Objectives

By the end of this notebook you will be able to:
- Detect and handle missing values with `isnull`, `dropna`, `fillna`, and `interpolate`
- Find and remove duplicate rows
- Coerce columns to correct types with `astype`, `pd.to_datetime`, and `pd.to_numeric`
- Clean string columns using the `.str` accessor
- Detect outliers using the IQR and z-score methods
- Visualize missing data patterns with a heatmap

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

print(f'Pandas  version: {pd.__version__}')
print(f'NumPy   version: {np.__version__}')
print(f'Seaborn version: {sns.__version__}')

## 1. Creating a Messy Dataset

Real-world data arrives with missing values, inconsistent types, duplicated rows, bad strings, and outliers. We simulate all of these so you can practice cleaning them.

In [ ]:
np.random.seed(42)
n = 120

names = [
    '  Alice ', 'Bob', 'CAROL', 'dave', '  Eve', 'Frank  ',
    'Grace', 'Hank', 'Iris', 'Jack'
]

raw_data = {
    'customer_id':  list(range(1001, 1001 + n)),
    'name':         np.random.choice(names, n),
    'age':          np.where(
                        np.random.rand(n) < 0.07,   # 7% missing
                        np.nan,
                        np.random.randint(18, 75, n).astype(float)
                    ),
    'signup_date':  np.where(
                        np.random.rand(n) < 0.05,
                        None,
                        np.random.choice([
                            '2022-01-15', '2022-13-01',  # bad date
                            '2023-06-30', '2023-07-04',
                            '2024-11-11', 'not a date'  # bad date
                        ], n)
                    ),
    'revenue':      np.where(
                        np.random.rand(n) < 0.08,
                        np.nan,
                        np.round(np.random.exponential(200, n), 2)
                    ),
    'region':       np.where(
                        np.random.rand(n) < 0.06,
                        np.nan,
                        np.random.choice(['North', 'South', 'east', 'WEST', ' East'], n)
                    ),
    'score':        np.where(
                        np.random.rand(n) < 0.1,
                        'N/A',
                        np.random.randint(0, 100, n).astype(str)
                    )
}

# Inject duplicates
df_raw = pd.DataFrame(raw_data)
df_raw = pd.concat([df_raw, df_raw.iloc[[5, 22, 50]]], ignore_index=True)

# Inject outliers into revenue
outlier_idx = np.random.choice(len(df_raw), 4, replace=False)
df_raw.loc[outlier_idx, 'revenue'] = [5000, 8000, -150, 9999]

print(f'Raw dataset shape: {df_raw.shape}')
print('\nSample rows:')
print(df_raw.head(8).to_string())
print('\nData types:')
print(df_raw.dtypes)

## 2. Detecting Missing Data

`isnull()` returns a boolean DataFrame. `isna()` is an alias. We can sum these to count missing values per column.

In [ ]:
# Missing value counts
missing = df_raw.isnull().sum()
missing_pct = (df_raw.isnull().mean() * 100).round(1)

print('Missing values per column:')
summary = pd.DataFrame({'count': missing, 'percent': missing_pct})
print(summary[summary['count'] > 0].sort_values('percent', ascending=False))

# Visualize missing data as a heatmap
plt.figure(figsize=(10, 5))
sns.heatmap(
    df_raw.isnull(),
    yticklabels=False,
    cmap='viridis',
    cbar_kws={'label': 'Missing (1=True)'},
    linewidths=0
)
plt.title('Missing Data Heatmap (yellow = missing)')
plt.tight_layout()
plt.show()

## 3. Handling Missing Values

Choose your strategy based on the column's role: drop rows only when missing data is small and random; fill with a statistic when you want to preserve all rows; interpolate for ordered sequences.

In [ ]:
df = df_raw.copy()

# dropna: remove rows where ALL columns are null (rarely useful)
print('Before dropna on revenue:', df.shape)
df_no_missing_rev = df.dropna(subset=['revenue'])
print('After dropna on revenue: ', df_no_missing_rev.shape)

# fillna: fill age with median, region with mode
age_median = df['age'].median()
df['age'] = df['age'].fillna(age_median)
print(f'\nFilled age NaN with median: {age_median}')

region_mode = df['region'].dropna().mode()[0]
df['region'] = df['region'].fillna(region_mode)
print(f'Filled region NaN with mode: {region_mode!r}')

# fillna revenue with forward fill then backward fill
df['revenue'] = df['revenue'].fillna(method='ffill').fillna(method='bfill')

# interpolate: great for time-ordered numeric series
ts = pd.Series([1.0, np.nan, np.nan, 4.0, np.nan, 6.0])
print('\nInterpolation example:')
print('Original:     ', ts.tolist())
print('Linear interp:', ts.interpolate(method='linear').tolist())

print(f'\nMissing after cleaning: {df.isnull().sum().sum()}')

## 4. Removing Duplicates

`duplicated()` marks repeated rows. `drop_duplicates()` removes them, keeping by default the first occurrence.

In [ ]:
print(f'Rows before dedup: {len(df)}')
print(f'Duplicate rows found: {df.duplicated().sum()}')

# Show which rows are duplicates
dupes = df[df.duplicated(keep=False)]
print('\nDuplicate rows (showing all occurrences):')
print(dupes[['customer_id', 'name', 'revenue']].head(6))

# Remove duplicates
df = df.drop_duplicates()
print(f'\nRows after  dedup: {len(df)}')

# Drop duplicates based on a specific column
df_unique_customers = df.drop_duplicates(subset=['customer_id'], keep='first')
print(f'Unique by customer_id: {len(df_unique_customers)}')

## 5. Type Coercion

Columns imported from CSV often arrive as strings. We need to cast them to the correct types.

In [ ]:
# score column contains 'N/A' strings — use errors='coerce' to turn bad values into NaN
df['score'] = pd.to_numeric(df['score'], errors='coerce')
print('score dtype after to_numeric:', df['score'].dtype)
print('score NaN count:', df['score'].isna().sum())
df['score'] = df['score'].fillna(df['score'].median())

# signup_date: coerce bad dates to NaT
df['signup_date'] = pd.to_datetime(df['signup_date'], errors='coerce')
print('\nsignup_date dtype:', df['signup_date'].dtype)
print('signup_date NaT count:', df['signup_date'].isna().sum())

# Now we can extract date parts
df = df.dropna(subset=['signup_date'])
df['signup_year']  = df['signup_date'].dt.year
df['signup_month'] = df['signup_date'].dt.month
print('\nDate components sample:')
print(df[['signup_date', 'signup_year', 'signup_month']].head(5))

# astype
df['age'] = df['age'].astype(int)
print('\nage dtype after astype:', df['age'].dtype)

## 6. String Cleaning

The `.str` accessor exposes vectorized string methods that operate on every element of a Series without a Python loop.

In [ ]:
print('Raw name values (unique):')
print(df['name'].unique())

# Strip whitespace and normalize case
df['name'] = df['name'].str.strip().str.title()
print('\nCleaned name values (unique):')
print(sorted(df['name'].unique()))

# Clean region
df['region'] = df['region'].str.strip().str.title()
print('\nCleaned region values (unique):')
print(sorted(df['region'].unique()))

# String operations demo
example = pd.Series(['  Hello World  ', 'foo_bar_baz', 'DATA123science'])
print('\nString operations:')
print('strip + upper:', example.str.strip().str.upper().tolist())
print('replace _ -> space:', example.str.replace('_', ' ', regex=False).tolist())
print('contains digits:', example.str.contains(r'\d').tolist())
print('extract digits:', example.str.extract(r'(\d+)')[0].tolist())

## 7. Outlier Detection: IQR and Z-Score Methods

Outliers can skew statistics and degrade model performance. Two classic detection methods: **IQR** (robust, distribution-free) and **z-score** (assumes normality).

In [ ]:
rev = df['revenue'].dropna()

# --- IQR Method ---
Q1  = rev.quantile(0.25)
Q3  = rev.quantile(0.75)
IQR = Q3 - Q1
lower_iqr = Q1 - 1.5 * IQR
upper_iqr = Q3 + 1.5 * IQR

outliers_iqr = df[(df['revenue'] < lower_iqr) | (df['revenue'] > upper_iqr)]
print(f'IQR method: lower={lower_iqr:.1f}, upper={upper_iqr:.1f}')
print(f'Outliers detected (IQR): {len(outliers_iqr)}')
print(outliers_iqr[['customer_id', 'name', 'revenue']].to_string())

# --- Z-Score Method ---
z_scores = np.abs(stats.zscore(df['revenue'].dropna()))
outlier_mask = z_scores > 3
print(f'\nZ-score method (|z| > 3): {outlier_mask.sum()} outliers')

# --- Visualize ---
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].boxplot(rev, vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightblue'))
axes[0].set_title('Boxplot (IQR-based whiskers)')
axes[0].set_ylabel('Revenue ($)')

axes[1].hist(rev, bins=30, color='steelblue', edgecolor='white')
axes[1].axvline(lower_iqr, color='red', linestyle='--', label='IQR bounds')
axes[1].axvline(upper_iqr, color='red', linestyle='--')
axes[1].set_title('Revenue Distribution with IQR Bounds')
axes[1].set_xlabel('Revenue ($)')
axes[1].legend()

rev_clean = rev[(rev >= lower_iqr) & (rev <= upper_iqr)]
axes[2].hist(rev_clean, bins=30, color='seagreen', edgecolor='white')
axes[2].set_title('Revenue After Removing Outliers')
axes[2].set_xlabel('Revenue ($)')

plt.tight_layout()
plt.show()

# Cap rather than remove
df['revenue_capped'] = df['revenue'].clip(lower=max(0, lower_iqr), upper=upper_iqr)
print(f'\nRevenue stats before capping: max={df["revenue"].max():.0f}')
print(f'Revenue stats after  capping: max={df["revenue_capped"].max():.0f}')

## 8. Final Clean Dataset Summary

In [ ]:
print('=== Clean Dataset Summary ===')
print(f'Shape: {df.shape}')
print('\nData types:')
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum())
print('\nDescriptive statistics:')
print(df[['age', 'revenue', 'score']].describe().round(2))

## Practice Exercises

**Exercise 1 — Missing Value Strategy**
Generate a DataFrame with 200 rows and 5 numeric columns where 15% of values are randomly set to `NaN`. For each column, compute the missing percentage, then fill columns with <10% missing using the median and drop rows where any column with >=10% missing has a NaN. Report the final shape.

**Exercise 2 — String Normalization Pipeline**
Given a Series of messy email addresses like `['  ALICE@Gmail.com ', 'bob@YAHOO.COM  ', None, 'carol@ hotmail.com']`, write a cleaning pipeline that strips whitespace, lowercases everything, removes spaces inside the string, and filters out null values.

**Exercise 3 — Outlier Comparison**
On the cleaned `revenue` column, apply both IQR and z-score (threshold=2.5) methods and find rows flagged by one method but NOT the other. What does this tell you about the two methods?